# Notebook 20 ? Training dataset ablation

Notebook n?y ?o ?nh h??ng c?a **training dataset** b?ng duy nh?t m? h?nh **Cross-Attention 3 modalities** ?? ch?t: NTv1 DNA 1001 + ESM1b protein 501, bio_core + SpliceAI + geometry k=64, ??u v?o ref/alt concat, `k_radius=16`, v? seeds 5, 6, 7, 37, 42.

C?c cohort ???c so s?nh:

- Train1 ??y ?? v? Train2 ??y ?? (s? m?u kh?ng c?n b?ng).
- Train1 v? Train2 ???c stratified downsample ch?nh x?c v? s? m?u c?a Train3, gi? t? l? benign/pathogenic g?n nh?t c? th? theo s? nguy?n.
- Train3 l? baseline ?? ch?y trong Notebook 17; Notebook n?y ch? ??c l?i k?t qu? 5 seed, kh?ng train Train3.

T?t c? cohort d?ng chung validation v? b?n t?p test. Checkpoint, epoch state, metric v? prediction ???c ghi theo cohort/seed; ch?y l?i c?ng batch s? resume ho?c b? qua l??t ?? ho?n th?nh.

Run All s? chu?n b? token cache Train1/Train2 n?u c?n thi?u, t?o geometry k=64 theo ??ng training context, train c?c cohort m?i v? cu?i c?ng l?u mean/std theo t?ng test set.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib
import json
import os
import sys

import pandas as pd
import torch
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()),
    _cwd,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.fm_ablation_preparation import (
    build_model_configs,
    build_split_configs,
    extract_missing_token_caches,
    inspect_token_cache,
)
from core.module06_architecture_experiments.fm_sequence_length_ablation import (
    build_center_geometry_embeddings,
)
from core.module06_architecture_experiments.training_dataset_ablation import (
    load_train3_reference_metrics,
    prepare_training_conditions,
    run_training_dataset_ablation,
    summarize_training_dataset_ablation,
)
from core.module03_geometry_extraction.geometry_v2 import GeometryV2Config, run_geometry_v2

# --- User-editable locations and execution controls ---
BASE_DIR = Path(r"F:\variant_data")
REFERENCE_BATCH_DIR = REPO_ROOT / "experiments" / "fusion_classifier_benchmark_20261120_192605"
RESUME_BATCH_RUN = True
BATCH_RUN_DIR_OVERRIDE = None  # Optional: set this to an explicit experiments/<batch> path.
BATCH_POINTER = REPO_ROOT / "experiments" / "_training_dataset_ablation_latest_batch.json"

if BATCH_RUN_DIR_OVERRIDE is not None:
    BATCH_RUN_DIR = Path(BATCH_RUN_DIR_OVERRIDE).resolve()
elif RESUME_BATCH_RUN and BATCH_POINTER.is_file():
    BATCH_RUN_DIR = Path(json.loads(BATCH_POINTER.read_text(encoding="utf-8"))["batch_run_dir"]).resolve()
else:
    BATCH_RUN_DIR = REPO_ROOT / "experiments" / f"training_dataset_ablation_{datetime.now():%Y%m%d_%H%M%S}"
    BATCH_POINTER.parent.mkdir(parents=True, exist_ok=True)
    BATCH_POINTER.write_text(
        json.dumps({"batch_run_dir": str(BATCH_RUN_DIR.resolve())}, indent=2),
        encoding="utf-8",
    )
BATCH_RUN_DIR.mkdir(parents=True, exist_ok=True)

GEOMETRY_K = 64
GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
GEOMETRY_NORMALIZATION_ROOT = "normalization_artifacts_v2_step2_matched"
GEOMETRY_INDEX_ROOT = "faiss_indexes_v2_step2_matched"
GEOMETRY_EMBEDDING_ROOT = BASE_DIR / "fm_embeddings" / "step2_geometry_center"
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
POOLED_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "sequence_ablation"
FM_PROFILE_JSON = BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"
GEOM_FEATURE_NAMES = [
    f"{modality}_{feature}"
    for modality in ("dna", "prot")
    for feature in ("LVD_L2", "LVD_Cosine", "LID", "LVD_Relative")
]
SEEDS = [5, 6, 7, 37, 42]
DOWNSAMPLE_RANDOM_STATE = 20261125  # Fixed across model seeds; change only for a new batch.
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DNA_MODEL_NAME = "nt_v1_500m_dna_1001"
PROTEIN_MODEL_NAME = "esm1b_650m_prot_501"
MODEL_CONFIGS = build_model_configs(dna_length=1001, protein_length=501)
DNA_MODEL = next(model for model in MODEL_CONFIGS if model["name"] == DNA_MODEL_NAME)
PROTEIN_MODEL = next(model for model in MODEL_CONFIGS if model["name"] == PROTEIN_MODEL_NAME)

FIXED_MODEL_CONFIG = {
    "attention_dropout": 0.18776012701077605,
    "dropout": 0.1955626033738947,
    "early_stop_patience": 6,
    "feature_dropout": 0.2507543060224074,
    "ffn_multiplier": 4,
    "k_radius": 16,
    "learning_rate": 0.00012206472463506827,
    "lr_factor": 0.5,
    "lr_patience": 3,
    "max_epochs": 30,
    "model_dim": 512,
    "num_heads": 8,
    "num_layers": 3,
    "pooling": "mean",
    "weight_decay": 3.380881049679085e-06,
    "batch_size": BATCH_SIZE,
}

TARGET_CONFIGS = [{
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "geom_feature_names": GEOM_FEATURE_NAMES,
    "geometry_dna_name": DNA_MODEL_NAME,
    "geometry_protein_name": PROTEIN_MODEL_NAME,
    "geometry_root": GEOMETRY_ROOT_NAME,
}]
TEST_SPLITS = ("test", "clinvarhq", "proteingym", "uniprot")
TRAINING_SPLITS = [
    {"name": split, "input_path": str(BASE_DIR / f"{split}_full_seq_final.parquet")}
    for split in ("train1", "train2")
]
STANDARD_SPLITS = build_split_configs(BASE_DIR)
ALL_SPLITS = TRAINING_SPLITS + STANDARD_SPLITS
ALL_SPLIT_NAMES = [split["name"] for split in ALL_SPLITS]

PIPELINE_CONFIG = {
    "batch_size": BATCH_SIZE,
    "epochs": FIXED_MODEL_CONFIG["max_epochs"],
    "lr": FIXED_MODEL_CONFIG["learning_rate"],
    "weight_decay": FIXED_MODEL_CONFIG["weight_decay"],
    "lr_factor": FIXED_MODEL_CONFIG["lr_factor"],
    "lr_patience": FIXED_MODEL_CONFIG["lr_patience"],
    "early_stop_patience": FIXED_MODEL_CONFIG["early_stop_patience"],
    "num_workers": 0,
    "seed": SEEDS[0],
    "xgb_n_jobs": 1,
}
PIPELINE = FusionBatchPipeline(
    base_dir=str(BASE_DIR),
    config=PIPELINE_CONFIG,
    datasets=[
        {"name": f"Train3Val_{name.title() if name != 'test' else 'Test'}", "train": "train3", "val": "val", "test": name}
        for name in TEST_SPLITS
    ],
    models_space=[DNA_MODEL, PROTEIN_MODEL],
    pooling_strategies=["center"],
    experiments=[{"name": "CrossAttention_ThreeModalities", "type": "pytorch", "fusion": "cross_attention"}],
    explainability={"enable_shap": False, "enable_lime": False},
    fm_profile_json=str(FM_PROFILE_JSON),
    geom_profile_json=None,
    geometry_dir=str(BASE_DIR / GEOMETRY_ROOT_NAME / "{context}"),
    geom_feature_names=GEOM_FEATURE_NAMES,
    batch_run_dir=str(BATCH_RUN_DIR),
    embedding_dir=str(POOLED_CACHE_ROOT),
)

print("Batch run:", BATCH_RUN_DIR)
print("Reference Train3 batch:", REFERENCE_BATCH_DIR)
print("Device:", DEVICE, "| batch size:", BATCH_SIZE, "| seeds:", SEEDS)
print("Fixed model:", DNA_MODEL_NAME, "+", PROTEIN_MODEL_NAME, "| ref/alt concat | k_radius=16")

c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batch run: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218
Reference Train3 batch: D:\Missense_variant_pathogenicity_prediction\experiments\fusion_classifier_benchmark_20261120_192605
Device: cuda | batch size: 64 | seeds: [5, 6, 7, 37, 42]
Fixed model: nt_v1_500m_dna_1001 + esm1b_650m_prot_501 | ref/alt concat | k_radius=16


## 1. Ki?m tra cohort v? n?p baseline Train3

Cell n?y t?o m?t t?p ID downsample c? ??nh cho Train1 v? Train2. T?p m?u ???c ch?n ??c l?p v?i seed hu?n luy?n ?? c?c seed ch? ?o dao ??ng kh?i t?o/training, kh?ng tr?n th?m dao ??ng do ch?n m?u. Train3 ch? ???c ??c t? k?t qu? Notebook 17.

In [2]:
ABLATION_ROOT = BATCH_RUN_DIR / "module06_architecture_experiments" / "training_dataset_ablation"
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

TRAINING_CONDITIONS, TRAINING_COHORT_INVENTORY, TRAINING_SAMPLING_MANIFEST = prepare_training_conditions(
    processed_dir=BASE_DIR / "processed_parquet",
    output_root=ABLATION_ROOT,
    downsample_random_state=DOWNSAMPLE_RANDOM_STATE,
)
TRAIN3_N = int(TRAINING_SAMPLING_MANIFEST["equal_sample_target"])
TRAIN3_CLASS_COUNTS = TRAINING_SAMPLING_MANIFEST["train3_reference_class_counts"]
print("Training cohort counts and class distributions:")
display(TRAINING_COHORT_INVENTORY)

REFERENCE_VAL, REFERENCE_TEST = load_train3_reference_metrics(
    reference_batch_dir=REFERENCE_BATCH_DIR,
    expected_seeds=SEEDS,
    train3_sample_count=TRAIN3_N,
    train3_class_counts=TRAIN3_CLASS_COUNTS,
    expected_model_config={key: value for key, value in FIXED_MODEL_CONFIG.items() if key != "batch_size"},
    expected_input_data_signature={
        "geometry_k": GEOMETRY_K,
        "geometry_root": GEOMETRY_ROOT_NAME,
        "geometry_features": GEOM_FEATURE_NAMES,
        "token_models": [DNA_MODEL_NAME, PROTEIN_MODEL_NAME],
        "target_config": "B_with_geometry",
        "embedding_representation": "ref_alt_concat",
    },
)
print("Reused Train3 baseline rows ? validation:", len(REFERENCE_VAL), "| test:", len(REFERENCE_TEST))
display(REFERENCE_TEST[["Dataset", "Seed", "MCC", "AUROC", "AUPRC"]].sort_values(["Dataset", "Seed"]))

Training cohort counts and class distributions:


,Training_Cohort,Training_Split,Sampling_Mode,Training_N,Benign_N,Pathogenic_N,Benign_Pct,Pathogenic_Pct,Source_N,Status
0,Train1 full,train1,full_dataset,104385,79652,24733,0.763060,0.236940,104385,train
1,Train1 stratified to Train3 N,train1,stratified_downsample_to_train3_count,23208,17709,5499,0.763056,0.236944,104385,train
2,Train2 full,train2,full_dataset,67990,44162,23828,0.649537,0.350463,67990,train
3,Train2 stratified to Train3 N,train2,stratified_downsample_to_train3_count,23208,15074,8134,0.649517,0.350483,67990,train
4,Train3 full (Notebook 17 reference),train3,existing_reference_no_rerun,23208,11604,11604,0.500000,0.500000,23208,reuse_notebook17


Reused Train3 baseline rows ? validation: 5 | test: 20


,Dataset,Seed,MCC,AUROC,AUPRC
401,Train3Val_ClinVarHQ,5,0.8788,0.9899,0.9901
417,Train3Val_ClinVarHQ,6,0.8664,0.9884,0.9894
433,Train3Val_ClinVarHQ,7,0.8792,0.9894,0.9893
449,Train3Val_ClinVarHQ,37,0.8943,0.9894,0.9887
465,Train3Val_ClinVarHQ,42,0.8867,0.9896,0.9902
402,Train3Val_ProteinGym,5,0.6963,0.9265,0.9191
418,Train3Val_ProteinGym,6,0.6718,0.9072,0.8802
434,Train3Val_ProteinGym,7,0.6942,0.9307,0.9216
450,Train3Val_ProteinGym,37,0.6745,0.9236,0.9133
466,Train3Val_ProteinGym,42,0.6702,0.9182,0.9064


## 2. Chu?n b? token cache Train1/Train2

Train1/Train2 ch?a c? token-level cache d?ng cho Cross-Attention. Cell tr?ch xu?t ref v? alt t? c?c chu?i 1001/501, l?u theo shard v? c? th? resume. Index cache Train3/validation/test ?? c? t? Notebook 17 s? ???c gh?p v?i hai split m?i.
On current files, Train1 has 104,385 rows, Train2 has 67,990, and Train3 has 23,208. Token extraction stores full hidden-state shards before deriving the k=16 window cache; using the existing Train3 cache size as a rough estimate, the new Train1/Train2 full-token shards may need about 650 GB combined, plus roughly 55 GB for compact k=16 window caches. These are linear estimates from the existing Train3 cache. The cache is shared under the FM cache root and can be reused by later batches.


In [3]:
REFERENCE_TOKEN_INDEX_PATH = (
    REFERENCE_BATCH_DIR / "module06_architecture_experiments" / "fm_ablation_token_cache_index.json"
)
if not REFERENCE_TOKEN_INDEX_PATH.is_file():
    raise FileNotFoundError(f"Notebook 17 token cache index is missing: {REFERENCE_TOKEN_INDEX_PATH}")

os.environ["HF_HUB_OFFLINE"] = "1"
TOKEN_CACHE_EXTENSION = extract_missing_token_caches(
    model_configs=[DNA_MODEL, PROTEIN_MODEL],
    split_configs=TRAINING_SPLITS,
    cache_root=TOKEN_CACHE_ROOT,
    batch_run_dir=BATCH_RUN_DIR,
    device=str(DEVICE),
    strict_context=True,
)
if TOKEN_CACHE_EXTENSION.get("status") != "complete":
    raise RuntimeError("Train1/Train2 token cache extraction is incomplete; rerun this cell to resume.")

TOKEN_CACHE_INDEX = json.loads(REFERENCE_TOKEN_INDEX_PATH.read_text(encoding="utf-8"))
for model_name, split_index in TOKEN_CACHE_EXTENSION["index"]["models"].items():
    TOKEN_CACHE_INDEX.setdefault("models", {}).setdefault(model_name, {}).update(split_index)
TOKEN_CACHE_INDEX["cache_root"] = str(TOKEN_CACHE_ROOT.resolve())
TOKEN_CACHE_INDEX_PATH = ABLATION_ROOT / "fm_ablation_token_cache_index.json"
TOKEN_CACHE_INDEX["index_path"] = str(TOKEN_CACHE_INDEX_PATH.resolve())
TOKEN_CACHE_INDEX_PATH.write_text(json.dumps(TOKEN_CACHE_INDEX, indent=2), encoding="utf-8")

TOKEN_CACHE_AUDIT = pd.DataFrame(inspect_token_cache(TOKEN_CACHE_ROOT, [DNA_MODEL, PROTEIN_MODEL], ALL_SPLITS))
TOKEN_CACHE_AUDIT = TOKEN_CACHE_AUDIT[["Model", "Split", "Status", "Shards", "Path", "Input_Matches"]]
display(TOKEN_CACHE_AUDIT)
if (TOKEN_CACHE_AUDIT["Status"] != "complete").any() or (~TOKEN_CACHE_AUDIT["Input_Matches"]).any():
    raise RuntimeError("A requested token cache is missing or does not match its source parquet.")
print("Merged token index:", TOKEN_CACHE_INDEX_PATH)

[*] Đang nạp Tokenizer: InstaDeepAI/nucleotide-transformer-500m-human-ref
[*] Đang nạp Mô hình (Trọng số float16): InstaDeepAI/nucleotide-transformer-500m-human-ref


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 1272.59it/s]


[!] Da patch compatibility cho remote EsmModel: get_head_mask, get_extended_attention_mask.
[+] Nạp mô hình thành công!

[Token cache estimate] nt_v1_500m_dna_1001/train1: ~92.46 GiB; free space at cache root: 1868.58 GiB


Token embeddings: train1:   0%|          | 0/3263 [00:00<?, ?it/s]

[Token Mapping] model=InstaDeepAI/nucleotide-transformer-500m-human-ref | seq_type=dna | strategy=dna_token_diff_fallback


Token embeddings: train1: 100%|██████████| 3263/3263 [16:41<00:00,  3.26it/s]


... (9 outputs left out)

Token embeddings: train2: 100%|██████████| 4250/4250 [36:23<00:00,  1.95it/s]


[Token cache] esm1b_650m_prot_501/train2: 67990 samples, 4250 shards -> F:\variant_data\fm_embeddings\token_level_cache\module06\esm1b_650m_prot_501\train2


,Model,Split,Status,Shards,Path,Input_Matches
0,nt_v1_500m_dna_1001,train1,complete,3263,F:\variant_data\fm_embeddings\token_level_cach...,True
1,nt_v1_500m_dna_1001,train2,complete,2125,F:\variant_data\fm_embeddings\token_level_cach...,True
2,nt_v1_500m_dna_1001,train3,complete,11604,F:\variant_data\fm_embeddings\token_level_cach...,True
3,nt_v1_500m_dna_1001,val,complete,6851,F:\variant_data\fm_embeddings\token_level_cach...,True
4,nt_v1_500m_dna_1001,test,complete,3048,F:\variant_data\fm_embeddings\token_level_cach...,True
5,nt_v1_500m_dna_1001,clinvarhq,complete,352,F:\variant_data\fm_embeddings\token_level_cach...,True
6,nt_v1_500m_dna_1001,proteingym,complete,736,F:\variant_data\fm_embeddings\token_level_cach...,True
7,nt_v1_500m_dna_1001,uniprot,complete,667,F:\variant_data\fm_embeddings\token_level_cach...,True
8,esm1b_650m_prot_501,train1,complete,6525,F:\variant_data\fm_embeddings\token_level_cach...,True
9,esm1b_650m_prot_501,train2,complete,4250,F:\variant_data\fm_embeddings\token_level_cach...,True


Merged token index: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\fm_ablation_token_cache_index.json


## 3. T?o geometry k=64 theo ??ng training context

M?i model d?ng geometry ???c fit tr?n ch?nh training cohort c?a n?: Train1 context cho l??t Train1, Train2 context cho l??t Train2. Validation v? test v?n gi? c?ng split, ch? thay context geometry theo t?p train. C?ch n?y kh?p quy tr?nh Module 6 v? tr?nh d?ng geometry context c?a m?t cohort kh?c.
For the full-vs-equal-count comparison, both Train1 variants share the geometry computed from the complete Train1 reference context, and both Train2 variants share the complete Train2 reference context. This keeps precomputed geometry features fixed within each source split; the downsampling changes only which labeled rows train the neural model.


In [4]:
CENTER_GEOMETRY_INDEX = build_center_geometry_embeddings(
    model_configs=[DNA_MODEL, PROTEIN_MODEL],
    split_configs=ALL_SPLITS,
    token_cache_index=TOKEN_CACHE_INDEX,
    output_root=GEOMETRY_EMBEDDING_ROOT,
)
display(CENTER_GEOMETRY_INDEX.groupby(["Model", "Status"], as_index=False).size())

GEOMETRY_RUN_CONFIG = GeometryV2Config(
    base_dir=str(BASE_DIR),
    embedding_dir=str(GEOMETRY_EMBEDDING_ROOT),
    contexts=["train1", "train2"],
    splits=ALL_SPLIT_NAMES,
    models=[DNA_MODEL_NAME, PROTEIN_MODEL_NAME],
    poolings=["center"],
    k_neighbors=GEOMETRY_K,
    resume=True,
    geometry_root=GEOMETRY_ROOT_NAME,
    normalization_root=GEOMETRY_NORMALIZATION_ROOT,
    index_root=GEOMETRY_INDEX_ROOT,
)
GEOMETRY_RUN = run_geometry_v2(GEOMETRY_RUN_CONFIG)
display(GEOMETRY_RUN)

GEOMETRY_AUDIT = []
for _context in ("train1", "train2"):
    for _split in ALL_SPLIT_NAMES:
        for _model in (DNA_MODEL_NAME, PROTEIN_MODEL_NAME):
            _path = BASE_DIR / GEOMETRY_ROOT_NAME / _context / _split / f"{_model}_center_geom_v2_norm.parquet"
            GEOMETRY_AUDIT.append({"Context": _context, "Split": _split, "Model": _model,
                                   "Status": "ready" if _path.is_file() else "missing", "Path": str(_path)})
GEOMETRY_AUDIT = pd.DataFrame(GEOMETRY_AUDIT)
display(GEOMETRY_AUDIT)
if (GEOMETRY_AUDIT["Status"] != "ready").any():
    raise FileNotFoundError("Matched k=64 geometry is incomplete; rerun this cell to resume.")

[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/train3
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/val
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/test
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/clinvarhq
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/proteingym
[GEOM EMBEDDING RESUME] nt_v1_500m_dna_1001/uniprot
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/train3
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/val
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/test
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/clinvarhq
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/proteingym
[GEOM EMBEDDING RESUME] esm1b_650m_prot_501/uniprot


,Model,Status,size
0,esm1b_650m_prot_501,created,2
1,esm1b_650m_prot_501,reused,6
2,nt_v1_500m_dna_1001,created,2
3,nt_v1_500m_dna_1001,reused,6


[V2] context=train1 | config=nt_v1_500m_dna_1001_center
[FAISS] GPU APIs khong kha dung, fallback sang CPU mode
[*] Đang khởi tạo FAISS Index không gian 1280 chiều...
[*] Đang nạp 104385 vector vào CPU Index...
[+] Đã huấn luyện và lưu Global FAISS Index tại: F:\variant_data/faiss_indexes_v2_step2_matched/train1/nt_v1_500m_dna_1001_center.index

[+] Đã nạp thành công CPU Index từ: F:\variant_data/faiss_indexes_v2_step2_matched/train1/nt_v1_500m_dna_1001_center.index
[V2] context=train1 | config=esm1b_650m_prot_501_center
[FAISS] GPU APIs khong kha dung, fallback sang CPU mode
[*] Đang khởi tạo FAISS Index không gian 1280 chiều...
[*] Đang nạp 104385 vector vào CPU Index...
[+] Đã huấn luyện và lưu Global FAISS Index tại: F:\variant_data/faiss_indexes_v2_step2_matched/train1/esm1b_650m_prot_501_center.index

[+] Đã nạp thành công CPU Index từ: F:\variant_data/faiss_indexes_v2_step2_matched/train1/esm1b_650m_prot_501_center.index
[V2] context=train2 | config=nt_v1_500m_dna_1001_center
[F

,context,split,config,rows,output
0,train1,train1,nt_v1_500m_dna_1001_center,104385,F:\variant_data/geometry_v2_k64_step2_matched/...
1,train1,train2,nt_v1_500m_dna_1001_center,67990,F:\variant_data/geometry_v2_k64_step2_matched/...
2,train1,train3,nt_v1_500m_dna_1001_center,23208,F:\variant_data/geometry_v2_k64_step2_matched/...
3,train1,val,nt_v1_500m_dna_1001_center,13701,F:\variant_data/geometry_v2_k64_step2_matched/...
4,train1,test,nt_v1_500m_dna_1001_center,6095,F:\variant_data/geometry_v2_k64_step2_matched/...
5,train1,clinvarhq,nt_v1_500m_dna_1001_center,703,F:\variant_data/geometry_v2_k64_step2_matched/...
6,train1,proteingym,nt_v1_500m_dna_1001_center,1472,F:\variant_data/geometry_v2_k64_step2_matched/...
7,train1,uniprot,nt_v1_500m_dna_1001_center,1333,F:\variant_data/geometry_v2_k64_step2_matched/...
8,train1,train1,esm1b_650m_prot_501_center,104385,F:\variant_data/geometry_v2_k64_step2_matched/...
9,train1,train2,esm1b_650m_prot_501_center,67990,F:\variant_data/geometry_v2_k64_step2_matched/...


,Context,Split,Model,Status,Path
0,train1,train1,nt_v1_500m_dna_1001,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
1,train1,train1,esm1b_650m_prot_501,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
2,train1,train2,nt_v1_500m_dna_1001,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
3,train1,train2,esm1b_650m_prot_501,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
4,train1,train3,nt_v1_500m_dna_1001,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
5,train1,train3,esm1b_650m_prot_501,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
6,train1,val,nt_v1_500m_dna_1001,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
7,train1,val,esm1b_650m_prot_501,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
8,train1,test,nt_v1_500m_dna_1001,ready,F:\variant_data\geometry_v2_k64_step2_matched\...
9,train1,test,esm1b_650m_prot_501,ready,F:\variant_data\geometry_v2_k64_step2_matched\...


## 4. Train/resume c?c cohort m?i

Cell ch?y b?n cohort m?i ? n?m seed. M?i checkpoint ???c ch?n b?ng MCC validation sau khi t?i ?u threshold tr?n validation; sau ?? c?ng checkpoint ???c ??nh gi? tr?n Test, ClinVarHQ, ProteinGym v? UniProt. Train3 kh?ng ???c train l?i.

In [5]:
ABLATION_CONTEXT = {
    "pipeline": PIPELINE,
    "TARGET_CONFIGS": TARGET_CONFIGS,
    "DNA_MODEL": DNA_MODEL,
    "PROTEIN_MODEL": PROTEIN_MODEL,
    "DEVICE": DEVICE,
    "token_embedding_cache_index": TOKEN_CACHE_INDEX,
    "token_batch_size": BATCH_SIZE,
    "test_splits": TEST_SPLITS,
    "geometry_root_name": GEOMETRY_ROOT_NAME,
    "train3_reference_source": str(REFERENCE_BATCH_DIR.resolve()),
    "input_data_signature": {
        "geometry_k": GEOMETRY_K,
        "geometry_root": GEOMETRY_ROOT_NAME,
        "geometry_features": GEOM_FEATURE_NAMES,
        "token_models": [DNA_MODEL_NAME, PROTEIN_MODEL_NAME],
        "target_config": "B_with_geometry",
        "feature_set": "bio_core+spliceai+geom_k64",
        "embedding_representation": "ref_alt_concat",
        "token_window": "variant-centered radius 16; 33 tokens per allele",
    },
}

ABLATION_RESULTS = run_training_dataset_ablation(
    context=ABLATION_CONTEXT,
    conditions=TRAINING_CONDITIONS,
    seeds=SEEDS,
    fixed_model_config=FIXED_MODEL_CONFIG,
    output_root=ABLATION_ROOT,
    train3_reference_validation=REFERENCE_VAL,
    train3_reference_test=REFERENCE_TEST,
    display_fn=display,
)
print("Seed-level validation:", ABLATION_RESULTS["validation_path"])
print("Seed-level test metrics:", ABLATION_RESULTS["test_path"])
print("Seed-level test predictions:", ABLATION_RESULTS["prediction_path"])
print("Cohort checkpoints and resume state:", ABLATION_ROOT)

D:\Missense_variant_pathogenicity_prediction\core\module06_architecture_experiments\training_dataset_ablation.py:433: RuntimeWarning: No resume manifest exists for legacy stage module06_training_dataset_ablation in D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation; the current settings are being recorded, but prior settings cannot be verified.
  ensure_stage_signature(


[Training Dataset Ablation] Train1 full | seed=5 | N=104,385
[CrossAttention_ThreeModalities] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 104385 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train1: 100%|██████████| 3263/3263 [04:48<00:00, 11.30shard/s]


[Window cache] dna/train1: 104385 samples ready in 288.8s


Window cache protein/train1: 100%|██████████| 6525/6525 [09:57<00:00, 10.92shard/s]


[Window cache] protein/train1: 104385 samples ready in 597.6s
[CrossAttention_ThreeModalities] Train loader ready: 1632 batches in 894.7s
[CrossAttention_ThreeModalities] Preparing validation dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



... (1,003 outputs left out)

[Window cache] protein/uniprot: 1333 samples ready in 0.1s


[CrossAttention_ThreeModalities] Test complete: B_with_geometry / Train2Val_UniProt MCC=0.5177 AUROC=0.8087 AUPRC=0.8030


{'condition': 'Train2 stratified to Train3 N',
 'seed': 42,
 'validation':                                       Run_ID Branch Evaluation_Split  \
 0  training_dataset_ablation_20261125_005218     DL       validation   
 
      Target_Config                         Ablation Pooling  \
 0  B_with_geometry  bio_core_dna_prot_spliceai_geom  center   
 
              DNA_Model           Prot_Model                         Network  \
 0  nt_v1_500m_dna_1001  esm1b_650m_prot_501  CrossAttention_ThreeModalities   
 
                      Architecture  ... Val_AUROC Val_AUPRC  \
 0  CrossAttention_ThreeModalities  ...    0.9783     0.931   
 
        Training_Cohort           Training_Cohort_Name Training_Split  \
 0  train2_equal_train3  Train2 stratified to Train3 N         train2   
 
                   Training_Sampling_Mode  Training_N_Samples  \
 0  stratified_downsample_to_train3_count               23208   
 
    Training_Benign_N  Training_Pathogenic_N  \
 0              15074         

Seed-level validation: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\seed_level_validation_metrics.csv
Seed-level test metrics: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\seed_level_test_metrics.csv
Seed-level test predictions: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\seed_level_test_predictions.csv
Cohort checkpoints and resume state: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation


## 5. T?ng h?p k?t qu?

B?ng test ???c t?ng h?p **ri?ng theo cohort v? t?ng t?p test**; m?i mean/std t?nh tr?n n?m seed c?a ch?nh model/cohort ??, kh?ng g?p c?c cohort v?i nhau. File seed-level v? profiling chi ti?t v?n ???c gi? ?? truy nguy?n t?ng l??t.

In [6]:
ABLATION_SUMMARY = summarize_training_dataset_ablation(
    output_root=ABLATION_ROOT,
    display_fn=display,
)
print("Test mean/std:", ABLATION_SUMMARY["test_summary_path"])
print("Validation mean/std:", ABLATION_SUMMARY["validation_summary_path"])
print("Completion inventory:", ABLATION_SUMMARY["completion_path"])
display(pd.read_csv(ABLATION_ROOT / "training_cohort_inventory.csv"))

'Test metrics: mean ± standard deviation across seeds, per cohort and test dataset'

,Training_Cohort,Training_Cohort_Name,Training_N_Samples,Dataset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,...,E2E_GFLOPs/Sample_mean,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std
0,train1_equal_train3,Train1 stratified to Train3 N,23208,Train1Val_ClinVarHQ,0.95022,0.004396,0.98334,0.003309,0.91496,0.011222,...,820.8911,0.0,35.534,0.005477,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
4,train1_full,Train1 full,104385,Train1Val_ClinVarHQ,0.95022,0.004125,0.97348,0.008764,0.92474,0.009404,...,820.8911,0.0,35.542,0.013038,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
1,train1_equal_train3,Train1 stratified to Train3 N,23208,Train1Val_ProteinGym,0.83436,0.012999,0.76252,0.019399,0.92648,0.010315,...,820.8911,0.0,35.020,0.000000,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
5,train1_full,Train1 full,104385,Train1Val_ProteinGym,0.82976,0.010103,0.75740,0.015028,0.92322,0.003879,...,820.8911,0.0,35.040,0.012247,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
6,train1_full,Train1 full,104385,Train1Val_Test,0.94138,0.002426,0.89112,0.011174,0.89384,0.008505,...,820.8911,0.0,34.654,0.011402,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
2,train1_equal_train3,Train1 stratified to Train3 N,23208,Train1Val_Test,0.93942,0.001268,0.89548,0.009262,0.88022,0.010066,...,820.8911,0.0,34.632,0.004472,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
7,train1_full,Train1 full,104385,Train1Val_UniProt,0.76352,0.006799,0.74028,0.010502,0.80966,0.012805,...,820.8911,0.0,35.114,0.008944,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
3,train1_equal_train3,Train1 stratified to Train3 N,23208,Train1Val_UniProt,0.75422,0.004724,0.74298,0.009114,0.77500,0.013586,...,820.8911,0.0,35.102,0.004472,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
12,train2_full,Train2 full,67990,Train2Val_ClinVarHQ,0.94820,0.002587,0.97386,0.004003,0.92010,0.002391,...,820.8911,0.0,35.532,0.008367,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
8,train2_equal_train3,Train2 stratified to Train3 N,23208,Train2Val_ClinVarHQ,0.93826,0.004680,0.96366,0.008643,0.90978,0.015665,...,820.8911,0.0,35.532,0.004472,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0


'Validation metrics: mean ± standard deviation across seeds'

,Training_Cohort,Training_Cohort_Name,Training_N_Samples,Val_Accuracy_mean,Val_Accuracy_std,Val_Precision_mean,Val_Precision_std,Val_Recall_mean,Val_Recall_std,Val_Specificity_mean,Val_Specificity_std,Val_F1_Score_mean,Val_F1_Score_std,Val_MCC_mean,Val_MCC_std,Val_AUROC_mean,Val_AUROC_std,Val_AUPRC_mean,Val_AUPRC_std
1,train1_full,Train1 full,104385,0.95504,0.001124,0.88186,0.010606,0.88262,0.008745,0.97208,0.003072,0.88218,0.002099,0.85442,0.002799,0.98268,0.000228,0.93914,0.000921
3,train2_full,Train2 full,67990,0.95202,0.000898,0.87232,0.007672,0.87698,0.007657,0.96974,0.002329,0.87460,0.002088,0.84498,0.002560,0.98046,0.000321,0.93430,0.001893
0,train1_equal_train3,Train1 stratified to Train3 N,23208,0.95124,0.000541,0.88350,0.007412,0.85760,0.012533,0.97330,0.002316,0.87026,0.002922,0.84048,0.003026,0.97982,0.000901,0.93234,0.002746
2,train2_equal_train3,Train2 stratified to Train3 N,23208,0.94908,0.000973,0.87040,0.012017,0.86168,0.012074,0.96966,0.003628,0.86588,0.001894,0.83456,0.002277,0.97776,0.001462,0.92826,0.004442
4,train3_reference,Train3 full (Notebook 17 reference),23208,0.94564,0.000688,0.86552,0.010071,0.84696,0.010882,0.96890,0.003085,0.85602,0.001506,0.82270,0.001623,0.97544,0.000195,0.92260,0.000866


'Completion inventory'

,Training_Cohort,Completed_Seed_Dataset_Rows,Seeds,Test_Datasets
0,train1_equal_train3,20,5,4
1,train1_full,20,5,4
2,train2_equal_train3,20,5,4
3,train2_full,20,5,4
4,train3_reference,20,5,4


Test mean/std: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\test_mean_std_by_training_cohort_dataset.csv
Validation mean/std: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\validation_mean_std_by_training_cohort.csv
Completion inventory: D:\Missense_variant_pathogenicity_prediction\experiments\training_dataset_ablation_20261125_005218\module06_architecture_experiments\training_dataset_ablation\completion_inventory.csv


,Training_Cohort,Training_Split,Sampling_Mode,Training_N,Benign_N,Pathogenic_N,Benign_Pct,Pathogenic_Pct,Source_N,Status
0,Train1 full,train1,full_dataset,104385,79652,24733,0.763060,0.236940,104385,train
1,Train1 stratified to Train3 N,train1,stratified_downsample_to_train3_count,23208,17709,5499,0.763056,0.236944,104385,train
2,Train2 full,train2,full_dataset,67990,44162,23828,0.649537,0.350463,67990,train
3,Train2 stratified to Train3 N,train2,stratified_downsample_to_train3_count,23208,15074,8134,0.649517,0.350483,67990,train
4,Train3 full (Notebook 17 reference),train3,existing_reference_no_rerun,23208,11604,11604,0.500000,0.500000,23208,reuse_notebook17
